In [ ]:
# ==========================================================
# BLOQUE DE PORTABILIDAD: Colab + Drive + Local
# ==========================================================
import os
import sys

# 1. Detectar si estamos en Google Colab
EN_COLAB = 'google.colab' in sys.modules

# 2. CONFIGURACIÓN IMPORTANTE: Nombra tu carpeta en Drive así:
CARPETA_DRIVE = 'Proyecto_IA'

if EN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Entramos a la subcarpeta notebooks para que las rutas ../data y ../outputs funcionen igual que en tu PC
    ruta_notebooks = f'/content/drive/MyDrive/{CARPETA_DRIVE}/notebooks'
    os.chdir(ruta_notebooks)
    print(f"✅ Colab conectado. Directorio actual: {os.getcwd()} (Estás en la carpeta notebooks)")
else:
    print(f"✅ Ejecutando en Local. Asegúrate de estar en la carpeta notebooks del proyecto.")
    print(f"Directorio actual: {os.getcwd()}")

# -*- coding: utf-8 -*-
"""
 MODELO LSTM HÍBRIDO CON TENSORFLOW

FASE 3: DEEP LEARNING PARA PREDICCIÓN FINANCIERA
"""

# 1. IMPORTAR LIBRERÍAS
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, concatenate, Input
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
print("✅ Librerías cargadas")
print(f"GPU activa: {tf.config.list_physical_devices('GPU')}")

# 2. CARGAR DATOS
df = pd.read_csv('../data/dataset_procesado.csv', sep=';')
df['daily_return'] = (df['close'] - df['open']) / df['open']
df['intraday_volatility'] = (df['high'] - df['low']) / df['close']
df['volume_scaled'] = df['volume'] / df['volume'].max()
df['price_range'] = df['high'] - df['low']

print(f"✅ Dataset: {df.shape[0]} filas")

# 3. PREPARAR TEXTO
headlines = df['Headline_esp'].astype(str).values
tokenizer = Tokenizer(num_words=5000, oov_token='<OOV>')
tokenizer.fit_on_texts(headlines)
sequences = tokenizer.texts_to_sequences(headlines)
max_length = 20
X_text = pad_sequences(sequences, maxlen=max_length, padding='post', truncating='post')
print(f"✅ Texto: {X_text.shape}")

# 4. PREPARAR DATOS NUMÉRICOS
features_num = ['daily_return', 'intraday_volatility', 'volume_scaled', 'price_range']
scaler = StandardScaler()
X_num = scaler.fit_transform(df[features_num].values)
y = df['Label'].values
print(f"✅ Numéricos: {X_num.shape}")

# 5. DIVIDIR DATOS
X_text_train, X_text_test, X_num_train, X_num_test, y_train, y_test = train_test_split(
    X_text, X_num, y, test_size=0.2, random_state=42, stratify=y
)

# Validación (15% del entrenamiento)
split_idx = int(0.85 * len(X_text_train))
X_text_val = X_text_train[split_idx:]
X_num_val = X_num_train[split_idx:]
y_val = y_train[split_idx:]
X_text_train = X_text_train[:split_idx]
X_num_train = X_num_train[:split_idx]
y_train = y_train[:split_idx]

print(f"✅ Entrenamiento: {len(y_train)} | Validación: {len(y_val)} | Prueba: {len(y_test)}")

# 6. CONSTRUIR MODELO LSTM HÍBRIDO
# Rama de texto
text_input = tf.keras.Input(shape=(max_length,), name='text_input')
x = Embedding(5000, 64, input_length=max_length)(text_input)
x = LSTM(32, return_sequences=False)(x)
x = Dropout(0.2)(x)
text_output = Dense(16, activation='relu')(x)

# Rama numérica
num_input = tf.keras.Input(shape=(4,), name='num_input')
y = Dense(16, activation='relu')(num_input)
y = Dense(8, activation='relu')(y)
num_output = Dense(8, activation='relu')(y)

# Combinar
combined = concatenate([text_output, num_output])
z = Dense(16, activation='relu')(combined)
z = Dropout(0.2)(z)
output = Dense(1, activation='sigmoid', dtype='float32')(z)

model_lstm = tf.keras.Model(inputs=[text_input, num_input], outputs=output)
model_lstm.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("✅ Modelo LSTM Híbrido construido")
model_lstm.summary()

# 7. ENTRENAR
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

print("\n Entrenando modelo LSTM...")
history = model_lstm.fit(
    [X_text_train, X_num_train],
    y_train,
    epochs=50,
    batch_size=32,
    validation_data=([X_text_val, X_num_val], y_val),
    callbacks=[early_stop],
    verbose=1
)

# 8. EVALUAR
test_loss, test_acc = model_lstm.evaluate([X_text_test, X_num_test], y_test, verbose=0)
print(f"\n✅ Precisión en prueba: {test_acc:.4f}")

y_pred_lstm = (model_lstm.predict([X_text_test, X_num_test]) > 0.5).astype(int)
y_proba_lstm = model_lstm.predict([X_text_test, X_num_test]).flatten()

print("\n Reporte de clasificación:")
print(classification_report(y_test, y_pred_lstm, target_names=['Baja', 'Subida']))

# 9. MATRIZ DE CONFUSIÓN
cm = confusion_matrix(y_test, y_pred_lstm)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de Confusión - LSTM')
plt.tight_layout()
plt.savefig('../outputs/graficos/matriz_confusion_lstm.png', dpi=300)
plt.show()
print("✅ Gráfico guardado")

# 10. EVOLUCIÓN DEL ENTRENAMIENTO
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history.history['loss'], label='Entrenamiento')
ax1.plot(history.history['val_loss'], label='Validación')
ax1.set_title('Pérdida')
ax1.legend()
ax1.grid(True)
ax2.plot(history.history['accuracy'], label='Entrenamiento')
ax2.plot(history.history['val_accuracy'], label='Validación')
ax2.set_title('Precisión')
ax2.legend()
ax2.grid(True)
plt.tight_layout()
plt.savefig('../outputs/graficos/evolucion_lstm.png', dpi=300)
plt.show()
print("✅ Gráfico guardado")

# 11. GUARDAR MODELO
model_lstm.save('../models/modelo_lstm_hibrido.h5')
print("✅ Modelo guardado")

print("\n✅ PROYECTO COMPLETADO EXITOSAMENTE 🎉")